# Silver Layer - Data Cleaning & Enrichment

In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, minute, date_format

df_trips = spark.table("taxicatalog.taxi_project_schema.bronze_taxi_trips")
df_zones = spark.table("taxicatalog.taxi_project_schema.bronze_taxi_zones")

In [0]:
# Convert to proper timestamp
df_trips = df_trips.withColumn(
    "pickup_time",
    to_timestamp("tpep_pickup_datetime", "yy/MM/dd H:mm")
).withColumn(
    "dropoff_time",
    to_timestamp("tpep_dropoff_datetime", "yy/MM/dd H:mm")
)

In [0]:
# Clean data
df_trips_clean = df_trips.filter(
    col("pickup_time").isNotNull() &
    col("dropoff_time").isNotNull() &
    col("PULocationID").isNotNull() &
    col("DOLocationID").isNotNull() &
    (col("trip_distance") > 0) &
    (col("total_amount") > 0)
)

In [0]:
# Extract ONLY TIME features
df_trips_clean = df_trips_clean \
.withColumn("pickup_hour", hour("pickup_time")) \
.withColumn("pickup_minute", minute("pickup_time")) \
.withColumn("dropoff_hour", hour("dropoff_time")) \
.withColumn("pickup_time_only", date_format("pickup_time", "HH:mm:ss")) \
.withColumn("dropoff_time_only", date_format("dropoff_time", "HH:mm:ss"))

In [0]:
from pyspark.sql.functions import col

df_enriched = df_trips_clean.alias("trips") \
.join(
    df_zones.alias("pu"),
    col("trips.PULocationID") == col("pu.LocationID"),
    "left"
) \
.join(
    df_zones.alias("do"),
    col("trips.DOLocationID") == col("do.LocationID"),
    "left"
)

In [0]:
df_final = df_enriched.select(
    col("trips.*"),

    col("pu.ZoneName").alias("pickup_zone"),
    col("pu.Latitude").alias("pickup_latitude"),
    col("pu.Longitude").alias("pickup_longitude"),

    col("do.ZoneName").alias("dropoff_zone"),
    col("do.Latitude").alias("dropoff_latitude"),
    col("do.Longitude").alias("dropoff_longitude")
)

df_final.write.format("delta") \
.mode("overwrite") \
.option("mergeSchema", "true") \
.saveAsTable("taxicatalog.taxi_project_schema.silver_taxi_trips_enriched")

In [0]:
display(df_final)